# 🗄️ updateDB — Cập nhật & Xoá Dữ liệu ChromaDB

Notebook kiểm thử **trước khi chuyển thành module** chức năng:

| Chức năng | Mô tả |
|-----------|-------|
| **Ingest mới** | Nhận văn bản OCR → tiền xử lý → tách Điều → chunking → upsert vào ChromaDB |
| **Duplicate check** | Kiểm tra văn bản đã tồn tại trong DB chưa trước khi upsert |
| **Auto-detect bãi bỏ** | Quét văn bản mới xem có bãi bỏ/hết hiệu lực văn bản nào → xoá khỏi ChromaDB |
| **Xoá thủ công** | Nhập số hiệu văn bản hoặc điều khoản cụ thể để xoá |
| **Verify** | Kiểm tra trước/sau mỗi thao tác |

## ⚠️ Ghi chú quan trọng về ID

Pipeline này dùng **hai tầng ID**, phải khớp chính xác với các notebook khác:

```
TẦNG 1 — doc_id (per article, từ preprocessData.ipynb):
  Format : "{doc_no}__{article_clean}__{content_hash[:10]}"
  Ví dụ  : "30/2020/NĐ-CP__Điều_5._Quyền_và_t__a3f8c12b45"
  Tạo bởi: make_doc_id() trong notebook này
  Dùng bởi: chunk_article(), make_chroma_id()

TẦNG 2 — ChromaDB chunk ID (per chunk, từ embedAndIndex.ipynb):
  Format : SHA256[:24] của f"{doc_id}|{chunk_index}|{text[:200]}"
  Tạo bởi: make_chroma_id() trong notebook này
  Dùng bởi: ChromaDB upsert/delete
```

Nếu sửa công thức tạo ID ở bất kỳ đâu, phải sửa **đồng bộ** cả `preprocessData.ipynb` và `embedAndIndex.ipynb`.

> ⚠️ **Notebook này không modify BM25 index** — BM25 sẽ được rebuild thủ công khi cần thiết (xem Cell 13).


---
## 0. Imports & Config

In [52]:
import re
import json
import time
import hashlib
import unicodedata
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
print("✅ Imports OK")
print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

✅ Imports OK
   Device: cpu


In [53]:
# ============================================================
# CONFIG — chỉnh sửa ở đây nếu cần
# ============================================================
PROJECT_ROOT = Path(".").resolve()   # thay bằng đường dẫn project nếu cần
CHUNK_DIR    = PROJECT_ROOT / "dataset" / "chunks"
CHROMA_DIR   = PROJECT_ROOT / "dataset" / "chromadb"
MODEL_PATH   = PROJECT_ROOT / "models" / "embedding"

EMBED_MODEL_NAME = "intfloat/multilingual-e5-large-instruct"
EMBED_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EMBED_BATCH_SIZE = 32  # thấp hơn embedAndIndex vì chạy local

COLLECTION_NAMES = {
    "legal"   : "legal_chunks",
    "forms"   : "forms_chunks",
    "examples": "examples_chunks",
}

# Chunking params — giống chunkData.ipynb
LEGAL_MAX_WORDS     = 400
LEGAL_MIN_WORDS     = 30
LEGAL_OVERLAP_WORDS = 50

# Sanity check paths
for name, path in {"chromadb": CHROMA_DIR, "chunks": CHUNK_DIR}.items():
    status = "✅" if path.exists() else "❌ KHÔNG TÌM THẤY"
    print(f"  [{status}] {name}: {path}")

  [✅] chromadb: /Users/coding/RAG_Drafting/RAG/dataset/chromadb
  [✅] chunks: /Users/coding/RAG_Drafting/RAG/dataset/chunks


## 1. Khởi tạo ChromaDB & Embedding Model

In [54]:
# ── ChromaDB client ──────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

def get_collection(name: str) -> chromadb.Collection:
    return chroma_client.get_collection(name=name)

# Kiểm tra collections hiện có
print("📦 ChromaDB collections:")
for key, col_name in COLLECTION_NAMES.items():
    try:
        col = get_collection(col_name)
        print(f"  ✅ {col_name}: {col.count():,} chunks")
    except Exception as e:
        print(f"  ❌ {col_name}: {e}")

📦 ChromaDB collections:
  ✅ legal_chunks: 213,487 chunks
  ✅ forms_chunks: 10 chunks
  ✅ examples_chunks: 150 chunks


In [55]:
# ── Embedding model ──────────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBED_MODEL_NAME}")
print("(Có thể mất 1-2 phút lần đầu nếu chưa cache)")
t0 = time.time()

embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=EMBED_DEVICE,
    cache_folder=str(MODEL_PATH),
)

print(f"✅ Model loaded in {time.time()-t0:.1f}s")

def embed_passages(texts: List[str]) -> np.ndarray:
    """Embed passages để index (prefix 'passage: ')."""
    prefixed = [f"passage: {t}" for t in texts]
    return embed_model.encode(
        prefixed, batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=len(texts) > 10,
    )

# Smoke test
_v = embed_passages(["Điều 1. Phạm vi điều chỉnh."])
print(f"   Embed shape: {_v.shape}  ✅")

Loading embedding model: intfloat/multilingual-e5-large-instruct
(Có thể mất 1-2 phút lần đầu nếu chưa cache)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Model loaded in 10.5s
   Embed shape: (1, 1024)  ✅


---
## 2. Utility Functions — Dùng chung cho toàn notebook

In [56]:
# ============================================================
# 2.1 ID helpers
# ============================================================

def make_doc_id(so_hieu: str, article: str, content: str) -> str:
    """
    Tái tạo doc_id khớp chính xác với preprocessData.ipynb → make_legal_doc_id().

    Format: "{doc_no}__{article_clean}__{content_hash[:10]}"
    Ví dụ : "30/2020/NĐ-CP__Điều_5._Quyền_và_t__a3f8c12b45"

    ⚠️  Phải lọc string 'nan' (pandas NaN → str) ở CẢ HAI tham số,
        giống như preprocessData kiểm tra `doc_no.lower() != 'nan'`
        và `article.lower() != 'nan'`.
    """
    content_hash = hashlib.sha256(content.encode("utf-8")).hexdigest()[:10]

    # Lọc giống preprocessData: bỏ None, chuỗi rỗng và chuỗi 'nan'
    doc_no        = str(so_hieu or "").strip()
    article_clean = str(article  or "").strip()

    doc_no_valid     = bool(doc_no)        and doc_no.lower()        != "nan"
    article_valid    = bool(article_clean) and article_clean.lower() != "nan"

    if doc_no_valid and article_valid:
        base = f"{doc_no}__{article_clean[:80]}"
    elif doc_no_valid:
        base = doc_no
    else:
        return f"sha_{content_hash}"

    base_clean = re.sub(r"[^\w/\-.]+", "_", base)
    return f"{base_clean}__{content_hash}"


def make_chroma_id(doc_id: str, chunk_index: int, text: str) -> str:
    """
    Tái tạo ChromaDB chunk ID khớp chính xác với embedAndIndex.ipynb (_make_chunk_id).

    Format: SHA256[:24] của f"{doc_id}|{chunk_index}|{text[:200]}"

    ⚠️  PHẢI khớp với embedAndIndex.ipynb → _make_chunk_id():
        raw = f"{row['doc_id']}|{row['chunk_index']}|{str(row['text'])[:200]}"
        return hashlib.sha256(raw.encode()).hexdigest()[:24]
    """
    raw = f"{doc_id}|{chunk_index}|{str(text)[:200]}"
    return hashlib.sha256(raw.encode()).hexdigest()[:24]


# ── Kiểm tra ID consistency ───────────────────────────────────────────────────
# Verify make_doc_id khớp với preprocessData
_so_hieu  = "30/2020/NĐ-CP"
_article  = "Điều 1. Phạm vi điều chỉnh"
_content  = "Nghị định này quy định về công tác văn thư."
_doc_id   = make_doc_id(_so_hieu, _article, _content)
_chroma_id = make_chroma_id(_doc_id, 0, f"{_article}\n{_content}")

print("✅ ID helpers defined.")
print(f"   doc_id    : {_doc_id}")
print(f"   chroma_id : {_chroma_id}")
assert len(_chroma_id) == 24, f"❌ chroma_id phải dài 24 ký tự, got {len(_chroma_id)}"
assert "__" in _doc_id, "❌ doc_id phải có __ separator"
print("   ✅ Format OK")


# ============================================================
# 2.2 Metadata helpers
# ============================================================

def coerce_metadata(meta: Dict) -> Dict:
    """Chuyển metadata về kiểu ChromaDB-compatible."""
    clean: Dict = {}
    for k, v in meta.items():
        if v is None:
            continue
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        elif isinstance(v, (np.bool_,)):
            clean[k] = bool(v)
        elif isinstance(v, (list, dict)):
            clean[k] = json.dumps(v, ensure_ascii=False)
        elif isinstance(v, (str, int, float, bool)):
            clean[k] = v
        else:
            clean[k] = str(v)
    return clean


# ============================================================
# 2.3 Text utils
# ============================================================

def count_words(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0


def normalize_text(text: str) -> str:
    """Chuẩn hoá whitespace, newline, tab — giống preprocessData."""
    if not isinstance(text, str):
        return ""
    text = text.replace("\t", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[^\S\n]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_doc_no(s: str) -> str:
    """Chuẩn hoá số hiệu để so sánh."""
    return s.strip().upper().replace(" ", "")


print("✅ Utility functions defined.")

✅ ID helpers defined.
   doc_id    : 30/2020/NĐ-CP__Điều_1._Phạm_vi_điều_chỉnh__b7e09fe947
   chroma_id : 5c7236b4c976b789ce7223ea
   ✅ Format OK
✅ Utility functions defined.


---
## 2b. Chọn nguồn văn bản OCR đầu vào

Có 2 tùy chọn:
- **Option A**: Dùng văn bản hardcode mẫu có sẵn (để test nhanh)
- **Option B**: Load từ file `.txt` trên disk

Chỉnh `INPUT_MODE` bên dưới để chọn.

In [57]:
# ============================================================
# NGUỒN ĐẦU VÀO — chỉnh INPUT_MODE hoặc để AUTO
# ============================================================
# "hardcode" → dùng SAMPLE_OCR_HARDCODE bên dưới (test nhanh)
# "file"     → load từ TXT_FILE_PATH cố định
# "prompt"   → hỏi đường dẫn khi chạy (khuyến nghị)

INPUT_MODE = "file"   # ← đổi thành "hardcode" hoặc "file" nếu muốn

TXT_FILE_PATH = PROJECT_ROOT / "input" / "luat-134.txt"
TXT_ENCODING  = "utf-8"

SAMPLE_OCR_HARDCODE = """
CHÍNH PHỦ
________
Số: 01/2025/NĐ-CP

CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
________________________
Hà Nội, ngày 01 tháng 01 năm 2025

NGHỊ ĐỊNH
VỀ QUY ĐỊNH CHI TIẾT THI HÀNH MỘT SỐ ĐIỀU CỦA LUẬT BAN HÀNH VĂN BẢN QUY PHẠM PHÁP LUẬT

Chương I. QUY ĐỊNH CHUNG

Điều 1. Phạm vi điều chỉnh
Nghị định này quy định chi tiết khoản 2 Điều 14, khoản 5 Điều 19 của Luật Ban hành
văn bản quy phạm pháp luật số 80/2015/QH13.

Điều 2. Đối tượng áp dụng
Nghị định này áp dụng đối với:
a) Cơ quan, tổ chức nhà nước;
b) Cán bộ, công chức, viên chức;
c) Tổ chức, cá nhân có liên quan.

Điều 20. Hiệu lực thi hành
Nghị định này có hiệu lực thi hành kể từ ngày ký.
Bãi bỏ Nghị định số 34/2016/NĐ-CP ngày 14 tháng 5 năm 2016.
"""

# ── Load theo mode ────────────────────────────────────────────────────────
if INPUT_MODE == "prompt":
    _path_input = input("📂 Nhập đường dẫn file .txt (Enter để dùng sample hardcode): ").strip()
    if _path_input:
        TXT_FILE_PATH = Path(_path_input)
        INPUT_MODE = "file"
    else:
        INPUT_MODE = "hardcode"
        print("   → Không nhập, dùng văn bản hardcode mẫu.")

if INPUT_MODE == "file":
    if not TXT_FILE_PATH.exists():
        raise FileNotFoundError(
            f"❌ Không tìm thấy file: {TXT_FILE_PATH}\n"
            f"   Đặt file txt vào đường dẫn trên, hoặc đổi INPUT_MODE = 'hardcode'"
        )
    RAW_OCR_TEXT = TXT_FILE_PATH.read_text(encoding=TXT_ENCODING)
    file_size_kb = TXT_FILE_PATH.stat().st_size / 1024
    print(f"✅ Đã load từ file: {TXT_FILE_PATH}")
    print(f"   Kích thước : {file_size_kb:.1f} KB | {len(RAW_OCR_TEXT):,} ký tự")
    print("   Preview (200 ký tự đầu):")
    print("   " + "-" * 50)
    print("   " + RAW_OCR_TEXT[:200].replace("\n", "\n   "))
    print("   " + "-" * 50)
else:
    RAW_OCR_TEXT = SAMPLE_OCR_HARDCODE
    print(f"✅ Dùng văn bản hardcode mẫu ({len(RAW_OCR_TEXT):,} ký tự)")

# ── Auto-detect và confirm số hiệu ───────────────────────────────────────
# (extract_doc_header được định nghĩa ở Cell 3 — nếu chạy Cell này độc lập
#  trước Cell 3 thì CONFIRMED_SO_HIEU = '' và sẽ hỏi lại ở Cell 10)
_header_preview = extract_doc_header(RAW_OCR_TEXT) if 'extract_doc_header' in dir() else {}
_detected = _header_preview.get("so_hieu", "")

if _detected:
    print(f"\n🔍 Số hiệu phát hiện tự động: {_detected}")
    _confirm_no = input("   Xác nhận số hiệu? [Enter = giữ nguyên / nhập để sửa]: ").strip()
    CONFIRMED_SO_HIEU = _confirm_no if _confirm_no else _detected
    if _confirm_no:
        print(f"   → Đã sửa thành: {CONFIRMED_SO_HIEU}")
    else:
        print(f"   → Giữ nguyên: {CONFIRMED_SO_HIEU}")
else:
    print("\n⚠️  Không tự động nhận diện được số hiệu.")
    CONFIRMED_SO_HIEU = input("   Nhập số hiệu văn bản (ví dụ: 134/2025/QH15): ").strip()
    if not CONFIRMED_SO_HIEU:
        print("⚠️  Số hiệu trống — sẽ hỏi lại ở Cell 10 khi chạy pipeline.")
    else:
        print(f"   → Số hiệu đã nhập: {CONFIRMED_SO_HIEU}")


✅ Đã load từ file: /Users/coding/RAG_Drafting/RAG/input/luat-134.txt
   Kích thước : 66.0 KB | 50,856 ký tự
   Preview (200 ký tự đầu):
   --------------------------------------------------
   ﻿QUỐC HỘI
   __________
   Luật số: 134/2025/QH15
   CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
   Độc lập - Tự do - Hạnh phúc 
   ______________________
   
   
   LUẬT 
   TRÍ TUỆ NHÂN TẠO
   
          Căn cứ Hiến pháp nước Cộng hòa xã hộ
   --------------------------------------------------

🔍 Số hiệu phát hiện tự động: 134/2025/QH15
   → Giữ nguyên: 134/2025/QH15


---
## 3. Tiền xử lý & Phân tách Điều từ văn bản OCR

In [58]:
# Pattern nhận diện đầu Điều
ARTICLE_PATTERN = re.compile(
    r'(?:^|\n)\s*'
    r'(Điều\s+\d+[a-z]?\.?'
    r'(?:\s+[^\n]+)?)',
    re.MULTILINE | re.UNICODE
)

# Pattern nhận diện đầu Chương
CHAPTER_PATTERN = re.compile(
    r'(?:^|\n)\s*(Chương\s+[IVXLCDM]+\.?(?:\s+[^\n]+)?)',
    re.MULTILINE | re.UNICODE
)


def extract_doc_header(text: str) -> Dict[str, str]:
    """Trích metadata từ phần header văn bản OCR."""
    header = {"so_hieu": "", "ten_van_ban": "", "co_quan": "", "loai_vb": "KHÔNG RÕ"}
    lines = text[:2000].split("\n")

    # Fix: thêm [0-9] để match QH15, QH13, v.v.
    so_hieu_pat = re.compile(
        r'\b(\d{1,3}/\d{4}/(?:[A-ZĐÔƯĂƠ][A-ZĐÔƯĂƠ0-9\-]+(?:/[A-ZĐÔƯĂƠ0-9\-]+)*))',
        re.UNICODE
    )
    for line in lines:
        m = so_hieu_pat.search(line)
        if m:
            header["so_hieu"] = m.group(1)
            break

    loai_map = [
        (r'LUẬT', 'LUẬT'), (r'NGHỊ ĐỊNH', 'NGHỊ ĐỊNH'),
        (r'NGHỊ QUYẾT', 'NGHỊ QUYẾT'), (r'THÔNG TƯ', 'THÔNG TƯ'),
        (r'QUYẾT ĐỊNH', 'QUYẾT ĐỊNH'), (r'PHÁP LỆNH', 'PHÁP LỆNH'),
    ]
    for pat, loai in loai_map:
        if re.search(pat, text[:500], re.IGNORECASE):
            header["loai_vb"] = loai
            break

    for line in lines[:30]:
        line_clean = line.strip()
        if len(line_clean) > 20 and re.search(
            r'[VỀ|QUY ĐỊNH|HƯỚNG DẪN|BAN HÀNH]', line_clean, re.IGNORECASE
        ):
            header["ten_van_ban"] = line_clean[:200]
            break

    return header


def split_articles(text: str) -> List[Dict[str, str]]:
    """Tách văn bản OCR thành danh sách các Điều."""
    parts = ARTICLE_PATTERN.split(text)

    if len(parts) <= 1:
        return [{"article": "Toàn văn", "content": text.strip(),
                 "chapter_id": "", "chapter_name": ""}]

    articles = []
    current_chapter_id, current_chapter_name = "", ""

    # Bắt Chương đứng trước Điều đầu tiên
    m = CHAPTER_PATTERN.search(parts[0])
    if m:
        chap_text = m.group(1).strip()
        chap_num  = re.search(r'[IVXLCDM]+', chap_text)
        current_chapter_id   = chap_num.group(0) if chap_num else ""
        current_chapter_name = chap_text

    i = 1
    while i < len(parts) - 1:
        article_header = parts[i].strip()
        raw_content    = parts[i + 1]

        chapter_m = CHAPTER_PATTERN.search(raw_content)
        if chapter_m:
            chap_text = chapter_m.group(1).strip()
            chap_num  = re.search(r'[IVXLCDM]+', chap_text)
            current_chapter_id   = chap_num.group(0) if chap_num else ""
            current_chapter_name = chap_text
            raw_content = raw_content[:chapter_m.start()] + raw_content[chapter_m.end():]

        content = normalize_text(raw_content)
        if content:
            articles.append({
                "article"     : article_header,
                "content"     : content,
                "chapter_id"  : current_chapter_id,
                "chapter_name": current_chapter_name,
            })
        i += 2

    return articles


print("✅ Parsing functions defined.")

✅ Parsing functions defined.


## 4. Chunking — Article-aware + Clause-aware

In [59]:
CLAUSE_PATTERN = re.compile(
    r'(?:^|\n)'
    r'(\d{1,2}\. '
    r'|[a-zđ]\) '
    r'|[àáảãạăắặằẳẵâấậầẩẫ]\) '
    r'|- (?=\S)'
    r')',
    re.MULTILINE | re.UNICODE
)


def split_into_clauses(text: str) -> List[Tuple[str, str]]:
    parts = CLAUSE_PATTERN.split(text)
    if len(parts) <= 1:
        return [(text.strip(), "no_clause")]
    result = []
    pre = parts[0].strip()
    if pre:
        result.append((pre, "preamble"))
    i = 1
    while i < len(parts) - 1:
        label   = parts[i].strip()
        content = parts[i + 1]
        clause_text = (label + " " + content).strip()
        if clause_text:
            ctype = "numbered" if re.match(r'\d', label) else ("bullet" if label.startswith("-") else "lettered")
            result.append((clause_text, ctype))
        i += 2
    return result if result else [(text.strip(), "no_clause")]


def fixed_word_split(text: str, max_words: int, overlap_words: int) -> List[str]:
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap_words
    return chunks


def chunk_article(article_dict: Dict, so_hieu: str, loai_vb: str,
                  ten_vb: str, ministry: str = "") -> List[Dict]:
    """
    Chunking 1 Điều → list chunk dicts.

    Đồng bộ hoàn toàn với chunkData.ipynb (chunk_legal_row):
      - CASE 1: Điều ngắn → text = "{article}\n{content}" (không ngoặc vuông)
      - CASE 2: split → merge dùng "\n" + buffer-based logic + split_type per-chunk
      - CASE 2 multi-chunk: text = "[{article}]\n{seg_text}" (có ngoặc vuông)
      - CASE 2 total==1:    text = seg_text (không header)
    """
    article      = article_dict["article"]
    content      = article_dict["content"]
    chapter_id   = article_dict.get("chapter_id", "")
    chapter_name = article_dict.get("chapter_name", "")

    # doc_id khớp với preprocessData.ipynb
    doc_id = make_doc_id(so_hieu, article, content)

    meta_base = {
        "doc_id"         : doc_id,
        "source_doc_no"  : so_hieu,
        "ministry"       : ministry,
        "type_normalized": loai_vb,
        "doc_name"       : ten_vb,
        "chapter_id"     : chapter_id,
        "chapter_name"   : chapter_name,
        "article"        : article,
    }

    # CASE 1: Điều ngắn → 1 chunk (giống chunkData: text = article + "\n" + content)
    if count_words(content) <= LEGAL_MAX_WORDS:
        text = f"{article}\n{content}" if article else content
        return [{
            **meta_base,
            "chunk_index" : 0,
            "total_chunks": 1,
            "split_type"  : "article",
            "text"        : text,
            "word_count"  : count_words(text),
        }]

    # CASE 2: Điều dài → split theo khoản
    clauses = split_into_clauses(content)
    raw_segments: List[Tuple[str, str]] = []  # (text, split_type)

    if len(clauses) == 1 and clauses[0][1] == "no_clause":
        # Không tìm được khoản → fixed_word fallback
        for seg in fixed_word_split(content, LEGAL_MAX_WORDS, LEGAL_OVERLAP_WORDS):
            raw_segments.append((seg, "fixed_word"))
    else:
        for clause_text, clause_type in clauses:
            if count_words(clause_text) <= LEGAL_MAX_WORDS:
                stype = "clause" if clause_type != "preamble" else "preamble"
                raw_segments.append((clause_text, stype))
            else:
                for seg in fixed_word_split(clause_text, LEGAL_MAX_WORDS, LEGAL_OVERLAP_WORDS):
                    raw_segments.append((seg, "fixed_word"))

    # Merge chunks quá ngắn — buffer-based, dùng "\n" (giống chunkData)
    merged: List[Tuple[str, str]] = []
    buf_text = ""
    buf_type = ""

    for seg_text, seg_type in raw_segments:
        seg_text = seg_text.strip()
        if not seg_text:
            continue
        if buf_text and count_words(buf_text) < LEGAL_MIN_WORDS:
            buf_text = buf_text + "\n" + seg_text  # giữ type của đoạn đầu
        else:
            if buf_text:
                merged.append((buf_text, buf_type))
            buf_text = seg_text
            buf_type = seg_type

    if buf_text.strip():
        merged.append((buf_text.strip(), buf_type))

    # Prepend article header vào chunk con — giống chunkData v2
    article_header = f"[{article}]" if article else ""
    total = len(merged)
    results = []
    for idx, (seg_text, seg_type) in enumerate(merged):
        if total > 1 and article_header:
            text = f"{article_header}\n{seg_text}"
        else:
            text = seg_text
        results.append({
            **meta_base,
            "chunk_index" : idx,
            "total_chunks": total,
            "split_type"  : seg_type,   # per-chunk type (giống chunkData)
            "text"        : text,
            "word_count"  : count_words(text),
        })

    return results


print("✅ Chunking functions defined.")

✅ Chunking functions defined.


## 5. Kiểm tra Văn bản đã tồn tại trong DB

Trước khi upsert, kiểm tra xem văn bản (theo `source_doc_no`) đã có chunks trong ChromaDB chưa. Điều này giúp:
- **Tránh upsert trùng** mà không hay biết
- **Phát hiện cập nhật**: văn bản đã có nhưng nội dung thay đổi → cần quyết định upsert đè hay skip

In [60]:
# ============================================================
# 5.1 Kiểm tra văn bản đã tồn tại trong DB chưa
# ============================================================

def check_doc_exists(
    so_hieu: str,
    collection_key: str = "legal",
    verbose: bool = True,
) -> Dict:
    """
    Kiểm tra văn bản có số hiệu `so_hieu` đã tồn tại trong ChromaDB chưa.

    Returns:
        {
            "exists"        : bool,
            "chunk_count"   : int,     # số chunks hiện có
            "article_count" : int,     # số Điều khác nhau
            "articles"      : list,    # danh sách Điều (tối đa 10)
            "so_hieu"       : str,
        }
    """
    col = get_collection(COLLECTION_NAMES[collection_key])

    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )

    chunk_count = len(results["ids"])
    articles_seen = list(dict.fromkeys(
        m.get("article", "") for m in results["metadatas"]
    ))  # preserves insertion order, removes dups

    info = {
        "exists"       : chunk_count > 0,
        "chunk_count"  : chunk_count,
        "article_count": len(articles_seen),
        "articles"     : articles_seen[:10],
        "so_hieu"      : so_hieu,
    }

    if verbose:
        if info["exists"]:
            print(f"⚠️  Văn bản '{so_hieu}' ĐÃ TỒN TẠI trong DB:")
            print(f"   Số chunks : {chunk_count:,}")
            print(f"   Số Điều   : {len(articles_seen)}")
            print(f"   Điều mẫu  : {articles_seen[:3]}")
            print(f"   → Upsert sẽ CẬP NHẬT các chunks trùng ID, thêm chunks mới.")
            print(f"   → Dùng delete_by_doc_no() trước nếu muốn xoá sạch rồi import lại.")
        else:
            print(f"✅ Văn bản '{so_hieu}' CHƯA tồn tại trong DB → sẵn sàng ingest.")

    return info


def check_docs_batch(
    so_hieu_list: List[str],
    collection_key: str = "legal",
) -> pd.DataFrame:
    """
    Kiểm tra hàng loạt nhiều văn bản cùng lúc.
    Trả về DataFrame tóm tắt trạng thái từng văn bản.
    """
    rows = []
    for so_hieu in so_hieu_list:
        info = check_doc_exists(so_hieu, collection_key, verbose=False)
        rows.append({
            "so_hieu"      : so_hieu,
            "exists"       : info["exists"],
            "chunks_in_db" : info["chunk_count"],
            "articles_in_db": info["article_count"],
            "status"       : "⚠️  Đã có" if info["exists"] else "✅ Chưa có",
        })

    df = pd.DataFrame(rows)
    print("\n📊 Trạng thái văn bản trong DB:")
    print(df.to_string(index=False))
    n_exist = df["exists"].sum()
    print(f"\n   Đã tồn tại: {n_exist}/{len(df)} văn bản")
    return df


print("✅ Duplicate check functions defined.")

✅ Duplicate check functions defined.


In [61]:
# ── Demo: kiểm tra một số văn bản quan trọng ────────────────────────────────
demo_nos = [
    "30/2020/NĐ-CP",
    "45/2019/QH14",
    "01/2025/NĐ-CP",   # mới, chưa có
]
status_df = check_docs_batch(demo_nos)


📊 Trạng thái văn bản trong DB:
      so_hieu  exists  chunks_in_db  articles_in_db    status
30/2020/NĐ-CP    True            43              38 ⚠️  Đã có
 45/2019/QH14    True           231             220 ⚠️  Đã có
01/2025/NĐ-CP   False             0               0 ✅ Chưa có

   Đã tồn tại: 2/3 văn bản


### 5.2 Kiểm tra chunk-level duplicate (theo chroma_id)

Ngoài việc kiểm tra theo `source_doc_no`, cũng có thể kiểm tra xem từng chunk cụ thể (theo `chroma_id`) đã có trong DB chưa. Hữu ích khi chỉ muốn upsert những chunk thực sự mới/thay đổi.

In [62]:
def check_chunks_exist(
    chunks: List[Dict],
    collection_key: str = "legal",
) -> Dict:
    """
    Kiểm tra từng chunk trong danh sách đã tồn tại trong DB chưa
    bằng cách so sánh chroma_id.

    Trả về:
        {
            "existing_ids"   : list[str],  # chroma_id đã có trong DB
            "new_ids"        : list[str],  # chroma_id chưa có
            "new_chunks"     : list[Dict], # chunks chưa có → cần upsert
            "existing_count" : int,
            "new_count"      : int,
        }
    """
    col = get_collection(COLLECTION_NAMES[collection_key])

    # Tính chroma_id cho mỗi chunk
    id_to_chunk = {}
    for c in chunks:
        cid = make_chroma_id(c["doc_id"], c["chunk_index"], c["text"])
        id_to_chunk[cid] = c

    all_ids = list(id_to_chunk.keys())

    # ChromaDB get() với danh sách IDs để kiểm tra
    try:
        existing = col.get(ids=all_ids, include=[])  # include=[] → chỉ lấy IDs
        existing_ids = set(existing["ids"])
    except Exception:
        existing_ids = set()

    new_ids    = [cid for cid in all_ids if cid not in existing_ids]
    new_chunks = [id_to_chunk[cid] for cid in new_ids]

    print(f"📋 Chunk-level duplicate check:")
    print(f"   Tổng chunks cần upsert : {len(all_ids)}")
    print(f"   Đã tồn tại trong DB    : {len(existing_ids)} (sẽ bị overwrite khi upsert)")
    print(f"   Hoàn toàn mới          : {len(new_ids)}")

    return {
        "existing_ids"  : list(existing_ids),
        "new_ids"       : new_ids,
        "new_chunks"    : new_chunks,
        "existing_count": len(existing_ids),
        "new_count"     : len(new_ids),
    }


print("✅ Chunk-level duplicate check defined.")

✅ Chunk-level duplicate check defined.


---
## 6. Auto-detect Bãi bỏ / Hết hiệu lực

In [63]:
OBSOLETE_PATTERNS = [
    re.compile(r'bãi bỏ\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    re.compile(r'thay thế\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    re.compile(r'hết hiệu lực\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    re.compile(r'(?:hủy bỏ|đình chỉ thi hành)\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
    re.compile(r'không còn hiệu lực\s+(?:[^,;.\n]*?\s+)?số\s+([\d/a-zA-ZĐÔƯĂ\-]+)', re.IGNORECASE | re.UNICODE),
]

# ⚠️  Phải đồng bộ với NEVER_REMOVE trong obsoleteFilter.ipynb
NEVER_REMOVE = {
    "30/2020/NĐ-CP", "01/2011/QH13", "80/2015/QH13", "15/2020/QH14",
    "34/2016/NĐ-CP", "45/2019/QH14", "58/2014/QH13", "22/2008/QH12",
    "58/2010/QH12", "138/2020/NĐ-CP", "115/2020/NĐ-CP", "61/2018/NĐ-CP",
    "34/2019/NĐ-CP", "76/2015/QH13", "77/2015/QH13", "36/2018/QH14",
    "02/2011/QH13", "03/2011/QH13",
}


def detect_abolished_docs(full_text: str) -> List[Dict]:
    """Quét văn bản OCR tìm số hiệu bị bãi bỏ. Loại trừ NEVER_REMOVE."""
    found: Dict[str, Dict] = {}
    for pat in OBSOLETE_PATTERNS:
        for m in pat.finditer(full_text):
            so_hieu = normalize_doc_no(m.group(1))
            if not so_hieu or len(so_hieu) < 5:
                continue
            if so_hieu in {normalize_doc_no(x) for x in NEVER_REMOVE}:
                continue
            snippet = full_text[max(0, m.start()-30):m.end()+60].replace("\n", " ")
            if so_hieu not in found:
                found[so_hieu] = {"count": 0, "snippets": [], "pattern": pat.pattern[:50]}
            found[so_hieu]["count"] += 1
            found[so_hieu]["snippets"].append(snippet)

    return [
        {"so_hieu": k, "count": v["count"],
         "snippet": v["snippets"][0][:150], "pattern": v["pattern"]}
        for k, v in found.items() if v["count"] >= 1
    ]


print("✅ Obsolete detection defined.")

✅ Obsolete detection defined.


---
## 7. Xoá Dữ liệu khỏi ChromaDB

In [64]:
def delete_by_doc_no(so_hieu: str, dry_run: bool = True) -> Dict:
    """Xoá tất cả chunks thuộc văn bản có số hiệu `so_hieu`."""
    if normalize_doc_no(so_hieu) in {normalize_doc_no(x) for x in NEVER_REMOVE}:
        print(f"🛡️  {so_hieu} nằm trong NEVER_REMOVE — KHÔNG xoá.")
        return {"so_hieu": so_hieu, "found_ids": [], "deleted_count": 0, "dry_run": dry_run}

    col = get_collection(COLLECTION_NAMES["legal"])
    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )
    found_ids = results["ids"]

    print(f"🔍 Tìm kiếm: '{so_hieu}' → {len(found_ids)} chunks")
    if not found_ids:
        print("   Không có gì để xoá.")
        return {"so_hieu": so_hieu, "found_ids": [], "deleted_count": 0, "dry_run": dry_run}

    print("   Sample (3 chunks đầu):")
    for meta in results["metadatas"][:3]:
        print(f"     - {meta.get('article', '?')[:60]}")

    if dry_run:
        print(f"   [DRY RUN] Sẽ xoá {len(found_ids)} chunks — đặt dry_run=False để thực hiện.")
        return {"so_hieu": so_hieu, "found_ids": found_ids, "deleted_count": 0, "dry_run": True}

    deleted = 0
    for i in range(0, len(found_ids), 100):
        col.delete(ids=found_ids[i:i+100])
        deleted += len(found_ids[i:i+100])
    print(f"   ✅ Đã xoá {deleted} chunks của '{so_hieu}'")
    print("   ⚠️  BM25 index chưa được cập nhật — xem Cell 13 để rebuild.")
    return {"so_hieu": so_hieu, "found_ids": found_ids, "deleted_count": deleted, "dry_run": False}


def delete_by_article(so_hieu: str, article_query: str, dry_run: bool = True) -> Dict:
    """Xoá chunks thuộc Điều cụ thể của một văn bản."""
    col = get_collection(COLLECTION_NAMES["legal"])
    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )
    if not results["ids"]:
        print(f"❌ Không tìm thấy văn bản '{so_hieu}' trong ChromaDB.")
        return {"found_ids": [], "deleted_count": 0}

    # Word-boundary regex để tránh "Điều 5" khớp vào "Điều 50"
    try:
        article_pattern = re.compile(
            r'(?<![\d\w])' + re.escape(article_query.strip()) + r'(?![\d\w])',
            re.IGNORECASE | re.UNICODE,
        )
    except re.error:
        article_pattern = re.compile(re.escape(article_query.strip()), re.IGNORECASE | re.UNICODE)

    matched_ids = [
        (cid, meta.get("article", ""))
        for cid, meta in zip(results["ids"], results["metadatas"])
        if article_pattern.search(meta.get("article", ""))
    ]

    print(f"🔍 Tìm Điều '{article_query}' trong '{so_hieu}' → {len(matched_ids)} chunks")
    if not matched_ids:
        print("   Không khớp — các Điều có trong DB:")
        seen: set = set()
        for meta in results["metadatas"][:20]:
            art = meta.get("article", "")
            if art not in seen:
                print(f"     · {art[:70]}")
                seen.add(art)
        return {"found_ids": [], "deleted_count": 0}

    for cid, art in matched_ids:
        print(f"     - [{cid[:12]}...] {art[:60]}")

    if dry_run:
        print(f"   [DRY RUN] Sẽ xoá {len(matched_ids)} chunks.")
        return {"found_ids": [x[0] for x in matched_ids], "deleted_count": 0, "dry_run": True}

    ids_to_delete = [x[0] for x in matched_ids]
    col.delete(ids=ids_to_delete)
    print(f"   ✅ Đã xoá {len(ids_to_delete)} chunks")
    return {"found_ids": ids_to_delete, "deleted_count": len(ids_to_delete), "dry_run": False}


# Verify regex word-boundary
_pat = re.compile(r'(?<![\d\w])' + re.escape('Điều 5') + r'(?![\d\w])', re.UNICODE)
assert _pat.search('Điều 5. Phạm vi')
assert not _pat.search('Điều 50. Khác')
print("✅ Delete functions defined. Regex word-boundary OK.")

✅ Delete functions defined. Regex word-boundary OK.


---
## 8. Upsert Dữ liệu mới vào ChromaDB

In [65]:
LEGAL_META_COLS = [
    "doc_id", "source_doc_no", "ministry", "type_normalized",
    "doc_name", "chapter_id", "chapter_name", "article",
    "chunk_index", "total_chunks", "split_type", "word_count",
]


def upsert_chunks(
    chunks: List[Dict],
    upsert_batch: int = 64,
    only_new: bool = False,   # True = bỏ qua chunk đã tồn tại (không overwrite)
) -> Dict:
    """
    Embed và upsert list chunks vào collection legal_chunks.

    Args:
        chunks      : list dict từ chunk_article()
        upsert_batch: số chunks mỗi lần upsert
        only_new    : True → chỉ upsert chunks chưa có trong DB (tiết kiệm thời gian embed)

    Returns:
        {upserted, skipped, errors}
    """
    col = get_collection(COLLECTION_NAMES["legal"])

    # Optional: lọc chỉ upsert chunks thực sự mới
    if only_new:
        check = check_chunks_exist(chunks)
        chunks_to_upsert = check["new_chunks"]
        skipped = check["existing_count"]
        print(f"   only_new=True → upsert {len(chunks_to_upsert)} chunks mới, skip {skipped} đã có.")
    else:
        chunks_to_upsert = chunks
        skipped = 0

    if not chunks_to_upsert:
        print("   Không có chunk nào cần upsert.")
        return {"upserted": 0, "skipped": skipped, "errors": []}

    upserted = 0
    errors   = []

    for i in range(0, len(chunks_to_upsert), upsert_batch):
        batch = chunks_to_upsert[i:i+upsert_batch]
        texts = [c["text"] for c in batch]

        try:
            embeddings = embed_passages(texts)
        except Exception as e:
            errors.append(f"Embed batch {i}: {e}")
            continue

        ids = [
            make_chroma_id(c["doc_id"], c["chunk_index"], c["text"])
            for c in batch
        ]
        metadatas = [
            coerce_metadata({k: c.get(k) for k in LEGAL_META_COLS})
            for c in batch
        ]

        try:
            col.upsert(ids=ids, embeddings=embeddings.tolist(),
                       documents=texts, metadatas=metadatas)
            upserted += len(batch)
            print(f"   Batch {i//upsert_batch + 1}: upserted {len(batch)} chunks")
        except Exception as e:
            errors.append(f"Upsert batch {i}: {e}")

    if not errors and upserted > 0:
        print(f"\n   ✅ Đã upsert {upserted} chunks vào ChromaDB.")
        print("   ⚠️  BM25 index chưa được cập nhật — xem Cell 13 để rebuild.")

    return {"upserted": upserted, "skipped": skipped, "errors": errors}


print("✅ Upsert function defined.")

✅ Upsert function defined.


---
## 9. Pipeline Tổng thể — Ingest văn bản OCR mới

In [66]:
def process_new_document(
    ocr_text: str,
    ministry: str = "",
    dry_run_delete: bool = True,
    skip_upsert: bool = False,
    only_new_chunks: bool = False,     # Chỉ upsert chunks chưa tồn tại
    force_if_exists: bool = True,      # False = dừng nếu văn bản đã có trong DB
    manual_so_hieu: str = "",
    manual_loai_vb: str = "",
    manual_ten_van_ban: str = "",
) -> Dict:
    """
    Pipeline đầy đủ xử lý văn bản OCR mới.

    Thứ tự:
      1. Trích header (số hiệu, loại, tên)
      2. Kiểm tra văn bản đã tồn tại trong DB chưa  ← MỚI
      3. Tách Điều
      4. Chunking
      5. Detect văn bản bị bãi bỏ
      6. Upsert văn bản mới (TRƯỚC)
      7. Xoá văn bản bãi bỏ (SAU — chỉ khi upsert thành công)

    Args:
        force_if_exists  : False → dừng và cảnh báo nếu văn bản đã có trong DB
        only_new_chunks  : True → chỉ embed/upsert chunks chưa tồn tại (tối ưu cho update nhỏ)
    """
    report = {
        "header"            : {},
        "doc_already_exists": False,
        "articles_found"    : 0,
        "chunks_created"    : 0,
        "abolished_found"   : [],
        "delete_results"    : [],
        "upsert_result"     : {},
        "errors"            : [],
    }

    print("=" * 60)
    print("🚀 Bắt đầu xử lý văn bản OCR")
    print("=" * 60)

    # ── BƯỚC 1: Trích header ───────────────────────────────────────────────
    header = extract_doc_header(ocr_text)
    if manual_so_hieu:    header["so_hieu"]    = manual_so_hieu
    if manual_loai_vb:    header["loai_vb"]    = manual_loai_vb
    if manual_ten_van_ban: header["ten_van_ban"] = manual_ten_van_ban

    report["header"] = header
    print(f"\n📋 Header:")
    print(f"   Số hiệu : {header['so_hieu'] or '(chưa nhận diện — cần manual_so_hieu)'}")
    print(f"   Loại VB : {header['loai_vb']}")
    print(f"   Tên VB  : {header['ten_van_ban'][:80] or '(chưa nhận diện)'}")

    if not header["so_hieu"]:
        report["errors"].append("Không nhận diện được số hiệu — cần manual_so_hieu")
        print("\n⚠️  Không tìm thấy số hiệu — pipeline dừng lại.")
        return report

    # ── BƯỚC 2: Kiểm tra đã tồn tại trong DB chưa ─────────────────────────
    print(f"\n🔎 Kiểm tra tồn tại trong DB...")
    exists_info = check_doc_exists(header["so_hieu"], verbose=True)
    report["doc_already_exists"] = exists_info["exists"]

    if exists_info["exists"] and not force_if_exists:
        msg = (
            f"Văn bản '{header['so_hieu']}' đã có {exists_info['chunk_count']} chunks trong DB. "
            f"Đặt force_if_exists=True để upsert đè, hoặc delete trước rồi ingest lại."
        )
        report["errors"].append(msg)
        print(f"\n⛔ Pipeline dừng: {msg}")
        return report

    # ── BƯỚC 3: Tách Điều ─────────────────────────────────────────────────
    print(f"\n📖 Tách Điều...")
    articles = split_articles(ocr_text)
    report["articles_found"] = len(articles)
    print(f"   → {len(articles)} Điều")

    if not articles:
        report["errors"].append("Không tìm thấy Điều nào")
        return report

    # ── BƯỚC 4: Chunking ──────────────────────────────────────────────────
    print(f"\n✂️  Chunking...")
    all_chunks = []
    for art in articles:
        all_chunks.extend(chunk_article(
            art,
            so_hieu  = header["so_hieu"],
            loai_vb  = header["loai_vb"],
            ten_vb   = header["ten_van_ban"],
            ministry = ministry,
        ))
    report["chunks_created"] = len(all_chunks)
    wc = [c["word_count"] for c in all_chunks]
    print(f"   → {len(all_chunks)} chunks | min={min(wc)} max={max(wc)} mean={sum(wc)//len(wc)} từ")

    # ── BƯỚC 5: Detect bãi bỏ ────────────────────────────────────────────
    print(f"\n🔎 Phát hiện văn bản bị bãi bỏ...")
    abolished = detect_abolished_docs(ocr_text)
    report["abolished_found"] = abolished
    if not abolished:
        print("   Không phát hiện văn bản bị bãi bỏ.")
    else:
        print(f"   ⚠️  {len(abolished)} văn bản bị bãi bỏ:")
        for item in abolished:
            print(f"   📌 {item['so_hieu']} ({item['count']} lần đề cập)")

    # ── BƯỚC 6: Upsert văn bản mới (TRƯỚC khi xoá) ───────────────────────
    if skip_upsert:
        print(f"\n⏭️  skip_upsert=True — bỏ qua upsert.")
    else:
        print(f"\n📤 Upsert {len(all_chunks)} chunks...")
        upsert_result = upsert_chunks(all_chunks, only_new=only_new_chunks)
        report["upsert_result"] = upsert_result
        if upsert_result["errors"]:
            print(f"   ❌ Lỗi upsert: {upsert_result['errors']}")
            print("   ⛔ Dừng — KHÔNG xoá văn bản bãi bỏ để tránh mất dữ liệu.")
            report["errors"].extend(upsert_result["errors"])
            return report

    # ── BƯỚC 7: Xoá văn bản bãi bỏ (SAU khi upsert thành công) ──────────
    if abolished:
        print(f"\n🗑️  Xoá văn bản bãi bỏ (dry_run={dry_run_delete})...")
        for item in abolished:
            del_result = delete_by_doc_no(item["so_hieu"], dry_run=dry_run_delete)
            report["delete_results"].append(del_result)

    # ── Tóm tắt ───────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("✅ Xử lý hoàn tất")
    print("=" * 60)
    upserted_count = report.get("upsert_result", {}).get("upserted", 0)
    deleted_count  = sum(r.get("deleted_count", 0) for r in report["delete_results"])
    if upserted_count > 0 or deleted_count > 0:
        print(f"\n{'⚠️  ' * 3}NHẮC NHỞ BM25{'  ⚠️' * 3}")
        print(f"   Upsert: {upserted_count} chunks | Xoá: {deleted_count} chunks")
        print("   BM25 chưa được cập nhật — chạy Cell 13 để rebuild.")
    return report


print("✅ process_new_document() defined.")

✅ process_new_document() defined.


---
## 10. Chạy Pipeline với Văn bản đã load

Dùng `RAW_OCR_TEXT` đã được load từ Cell 2b (hardcode hoặc file).

In [67]:
def db_status() -> None:
    print("📊 Trạng thái ChromaDB hiện tại:")
    for key, col_name in COLLECTION_NAMES.items():
        try:
            col = get_collection(col_name)
            print(f"   {col_name}: {col.count():,} chunks")
        except Exception as e:
            print(f"   {col_name}: ❌ {e}")

db_status()

📊 Trạng thái ChromaDB hiện tại:
   legal_chunks: 213,487 chunks
   forms_chunks: 10 chunks
   examples_chunks: 150 chunks


In [69]:
# ============================================================
# Chạy pipeline — dry-run trước, double confirm để thực thi thật
# ============================================================

# Đảm bảo CONFIRMED_SO_HIEU tồn tại (fallback nếu Cell 2b bị bỏ qua)
if 'CONFIRMED_SO_HIEU' not in dir() or not CONFIRMED_SO_HIEU:
    CONFIRMED_SO_HIEU = input("⚠️  Nhập số hiệu văn bản (ví dụ: 134/2025/QH15): ").strip()
    if not CONFIRMED_SO_HIEU:
        raise ValueError("❌ Số hiệu không được để trống.")

# ── BƯỚC 1: Xem trước (luôn safe) ────────────────────────────────────────
print("=" * 60)
print("👁️  CHẾ ĐỘ XEM TRƯỚC (dry_run_delete=True, skip_upsert=True)")
print("=" * 60)

report_preview = process_new_document(
    ocr_text         = RAW_OCR_TEXT,
    ministry         = "Chính phủ",   # ← Sửa nếu cần
    dry_run_delete   = True,
    skip_upsert      = True,
    force_if_exists  = True,
    only_new_chunks  = False,
    manual_so_hieu   = CONFIRMED_SO_HIEU,
)

# ── BƯỚC 2: Double confirm ────────────────────────────────────────────────
print()
_so_hieu_final = report_preview["header"].get("so_hieu", "?")
_n_chunks      = report_preview.get("chunks_created", 0)
_n_abolished   = len(report_preview.get("abolished_found", []))
_already_exist = report_preview.get("doc_already_exists", False)
_abolished_nos = [x['so_hieu'] for x in report_preview.get('abolished_found', [])]

print("╔══════════════════════════════════════════════════════════╗")
print("║           ⚠️   XÁC NHẬN TRƯỚC KHI THỰC THI THẬT   ⚠️   ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Số hiệu upsert : {_so_hieu_final:<40}║")
print(f"║  Chunks sẽ thêm : {_n_chunks:<40}║")
print(f"║  Đã có trong DB : {'⚠️  CÓ (sẽ upsert đè)' if _already_exist else '✅ Chưa có':<40}║")
print(f"║  VB bãi bỏ (xoá): {str(_abolished_nos)[:40]:<40}║")
print("╚══════════════════════════════════════════════════════════╝")

if report_preview.get("errors"):
    print(f"\n❌ Lỗi ở bước xem trước: {report_preview['errors']}")
    print("   Kiểm tra lại input. Pipeline dừng — chưa có gì thay đổi.")
    report = report_preview
else:
    _answer = input("\n▶ Thực thi thật? Nhập ĐỒNG Ý để xác nhận (Enter để huỷ): ").strip()

    if _answer.upper() in ("ĐỒNG Ý", "DONG Y", "Y", "YES", "OK"):
        print("\n✅ Đã xác nhận — bắt đầu thực thi thật...")
        report = process_new_document(
            ocr_text         = RAW_OCR_TEXT,
            ministry         = "Chính phủ",   # ← Sửa nếu cần
            dry_run_delete   = False,
            skip_upsert      = False,
            force_if_exists  = True,
            only_new_chunks  = False,
            manual_so_hieu   = CONFIRMED_SO_HIEU,
        )
    else:
        print("\n⛔ Đã huỷ — không có thay đổi nào được thực hiện.")
        report = report_preview


👁️  CHẾ ĐỘ XEM TRƯỚC (dry_run_delete=True, skip_upsert=True)
🚀 Bắt đầu xử lý văn bản OCR

📋 Header:
   Số hiệu : 134/2025/QH15
   Loại VB : LUẬT
   Tên VB  : Luật số: 134/2025/QH15

🔎 Kiểm tra tồn tại trong DB...
✅ Văn bản '134/2025/QH15' CHƯA tồn tại trong DB → sẵn sàng ingest.

📖 Tách Điều...
   → 35 Điều

✂️  Chunking...
   → 42 chunks | min=34 max=418 mean=266 từ

🔎 Phát hiện văn bản bị bãi bỏ...
   Không phát hiện văn bản bị bãi bỏ.

⏭️  skip_upsert=True — bỏ qua upsert.

✅ Xử lý hoàn tất

╔══════════════════════════════════════════════════════════╗
║           ⚠️   XÁC NHẬN TRƯỚC KHI THỰC THI THẬT   ⚠️   ║
╠══════════════════════════════════════════════════════════╣
║  Số hiệu upsert : 134/2025/QH15                           ║
║  Chunks sẽ thêm : 42                                      ║
║  Đã có trong DB : ✅ Chưa có                               ║
║  VB bãi bỏ (xoá): []                                      ║
╚══════════════════════════════════════════════════════════╝

✅ Đã xác 

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

   Batch 1: upserted 42 chunks

   ✅ Đã upsert 42 chunks vào ChromaDB.
   ⚠️  BM25 index chưa được cập nhật — xem Cell 13 để rebuild.

✅ Xử lý hoàn tất

⚠️  ⚠️  ⚠️  NHẮC NHỞ BM25  ⚠️  ⚠️  ⚠️
   Upsert: 42 chunks | Xoá: 0 chunks
   BM25 chưa được cập nhật — chạy Cell 13 để rebuild.


In [70]:
print("\n📄 BÁO CÁO XỬ LÝ:")
print(f"  Số hiệu       : {report['header'].get('so_hieu', 'N/A')}")
print(f"  Loại VB       : {report['header'].get('loai_vb', 'N/A')}")
print(f"  Đã có trong DB: {'⚠️  CÓ' if report['doc_already_exists'] else '✅ Chưa có'}")
print(f"  Số Điều       : {report['articles_found']}")
print(f"  Số chunks tạo : {report['chunks_created']}")
print(f"  Upsert        : {report['upsert_result'].get('upserted', 0)} chunks")
print(f"  Skip (đã có)  : {report['upsert_result'].get('skipped', 0)} chunks")
print(f"  VB bãi bỏ     : {len(report['abolished_found'])}")
deleted_total = sum(r.get('deleted_count', 0) for r in report['delete_results'])
print(f"  Chunks đã xoá : {deleted_total}")
all_errors = report.get('errors', []) + report.get('upsert_result', {}).get('errors', [])
print(f"  Lỗi           : {all_errors if all_errors else 'Không có'}")


📄 BÁO CÁO XỬ LÝ:
  Số hiệu       : 134/2025/QH15
  Loại VB       : LUẬT
  Đã có trong DB: ✅ Chưa có
  Số Điều       : 35
  Số chunks tạo : 42
  Upsert        : 42 chunks
  Skip (đã có)  : 0 chunks
  VB bãi bỏ     : 0
  Chunks đã xoá : 0
  Lỗi           : Không có


---
## 11. Xoá Thủ công

In [ ]:
# ── Xoá theo số hiệu văn bản (input + double confirm) ────────────────────

_del_so_hieu = input("🗑️  Nhập số hiệu văn bản muốn xoá (ví dụ: 110/2004/NĐ-CP): ").strip()
if not _del_so_hieu:
    print("⛔ Số hiệu trống — huỷ.")
else:
    result_preview = delete_by_doc_no(so_hieu=_del_so_hieu, dry_run=True)
    _n_del = len(result_preview["found_ids"])

    if _n_del == 0:
        print(f"\n   '{_del_so_hieu}' không có trong DB — không cần xoá.")
    else:
        print(f"\n⚠️  Sẽ xoá {_n_del} chunks của '{_del_so_hieu}' khỏi ChromaDB.")
        _confirm2 = input("   Xác nhận xoá thật? Nhập ĐỒNG Ý để tiếp tục: ").strip()
        if _confirm2.upper() in ("ĐỒNG Ý", "DONG Y", "Y", "YES"):
            result_delete = delete_by_doc_no(so_hieu=_del_so_hieu, dry_run=False)
            print(f"\n✅ Đã xoá {result_delete['deleted_count']} chunks của '{_del_so_hieu}'.")
            print("   ⚠️  Nhớ rebuild BM25 ở Cell 13 nếu xoá > 1% tổng chunks.")
        else:
            print("\n⛔ Đã huỷ — không có gì bị xoá.")


In [ ]:
# ── Cell này đã được gộp vào cell trên ──────────────────────────────────
print("ℹ️  Xem cell trên để xoá theo số hiệu — đã tích hợp input + double confirm.")


In [ ]:
# ── Xoá theo Điều cụ thể (input + double confirm) ────────────────────────

_art_so_hieu = input("📋 Số hiệu văn bản (ví dụ: 30/2020/NĐ-CP): ").strip()
_art_query   = input("📋 Tên / số Điều cần xoá (ví dụ: Điều 5): ").strip()

if not _art_so_hieu or not _art_query:
    print("⛔ Thiếu thông tin — huỷ.")
else:
    result_art_preview = delete_by_article(
        so_hieu       = _art_so_hieu,
        article_query = _art_query,
        dry_run       = True,
    )
    _n_art = len(result_art_preview["found_ids"])

    if _n_art == 0:
        print(f"\n   Không tìm thấy Điều '{_art_query}' trong '{_art_so_hieu}'.")
    else:
        print(f"\n⚠️  Sẽ xoá {_n_art} chunks của Điều '{_art_query}' trong '{_art_so_hieu}'.")
        _confirm_art = input("   Xác nhận xoá thật? Nhập ĐỒNG Ý để tiếp tục: ").strip()
        if _confirm_art.upper() in ("ĐỒNG Ý", "DONG Y", "Y", "YES"):
            result_art = delete_by_article(
                so_hieu       = _art_so_hieu,
                article_query = _art_query,
                dry_run       = False,
            )
            print(f"\n✅ Đã xoá {result_art['deleted_count']} chunks.")
            print("   ⚠️  Nhớ rebuild BM25 ở Cell 13 nếu cần.")
        else:
            print("\n⛔ Đã huỷ — không có gì bị xoá.")


---
## 12. Xoá hàng loạt & Verify

In [ ]:
def batch_delete_docs(so_hieu_list: List[str], dry_run: bool = True) -> pd.DataFrame:
    """Xoá nhiều văn bản từ danh sách số hiệu."""
    rows = []
    print(f"{'[DRY RUN] ' if dry_run else ''}Xử lý {len(so_hieu_list)} văn bản...\n")
    for so_hieu in so_hieu_list:
        res = delete_by_doc_no(so_hieu, dry_run=dry_run)
        rows.append({
            "so_hieu"      : so_hieu,
            "found_chunks" : len(res["found_ids"]),
            "deleted"      : res["deleted_count"],
            "protected"    : normalize_doc_no(so_hieu) in {normalize_doc_no(x) for x in NEVER_REMOVE},
        })
    df = pd.DataFrame(rows)
    print("\n📊 Kết quả:")
    print(df.to_string(index=False))
    total_col = 'deleted' if not dry_run else 'found_chunks'
    print(f"\n   Tổng chunks {'đã' if not dry_run else 'sẽ bị'} xoá: {df[total_col].sum()}")
    return df


def verify_doc(so_hieu: str, show_articles: bool = True) -> None:
    """Xem thông tin một văn bản trong ChromaDB."""
    col = get_collection(COLLECTION_NAMES["legal"])
    results = col.get(
        where={"source_doc_no": {"$eq": so_hieu}},
        include=["metadatas"],
    )
    print(f"📋 Văn bản: {so_hieu}")
    print(f"   Tổng chunks: {len(results['ids'])}")
    if not results["ids"]:
        print("   (Không có trong DB)")
        return
    if show_articles:
        articles_seen: Dict[str, int] = {}
        for meta in results["metadatas"]:
            art = meta.get("article", "(không rõ)")
            articles_seen[art] = articles_seen.get(art, 0) + 1
        print(f"   Số Điều: {len(articles_seen)}")
        for art, cnt in list(articles_seen.items())[:20]:
            print(f"     {cnt:2d} chunk(s)  {art[:70]}")
        if len(articles_seen) > 20:
            print(f"     ... và {len(articles_seen)-20} Điều khác")


# ── Xoá hàng loạt (input + double confirm) ──────────────────────────────────
print("Nhập từng số hiệu văn bản cần xoá. Dòng trống để kết thúc.")
_batch_list = []
while True:
    _line = input(f"  [{len(_batch_list)+1}] Số hiệu (Enter để kết thúc): ").strip()
    if not _line:
        break
    _batch_list.append(_line)

if not _batch_list:
    print("⛔ Danh sách trống — huỷ.")
else:
    summary_df = batch_delete_docs(_batch_list, dry_run=True)
    _confirm_batch = input("\n▶ Xoá thật tất cả? Nhập ĐỒNG Ý để tiếp tục: ").strip()
    if _confirm_batch.upper() in ("ĐỒNG Ý", "DONG Y", "Y", "YES"):
        summary_df = batch_delete_docs(_batch_list, dry_run=False)
        print("\n✅ Hoàn tất. Nhớ rebuild BM25 ở Cell 13 nếu cần.")
    else:
        print("\n⛔ Đã huỷ.")

print("\n" + "-" * 40)
_verify_no = input("Nhập số hiệu để verify (Enter để bỏ qua): ").strip()
if _verify_no:
    verify_doc(_verify_no)

In [ ]:
db_status()

---
## 13. (Tuỳ chọn) Rebuild BM25 Index

> ⚠️ Chạy cell này **sau khi đã upsert/xoá xong** để đồng bộ BM25 với ChromaDB.

In [71]:
print("""📋 Hướng dẫn rebuild BM25 sau khi cập nhật ChromaDB:

  OPTION 1 — Rebuild qua hybrid_retrieval.py (khuyến nghị):
    from hybrid_retrieval import init_retriever
    init_retriever(force_rebuild_bm25=True)

  OPTION 2 — Xoá cache thủ công rồi chạy lại:
    Xoá file .pkl → lần chạy tiếp tự detect và rebuild.

  Khi nào cần rebuild ngay?
    - Upsert / xoá > 1% tổng chunks (~2,400 chunks): rebuild ngay.
    - Ít hơn: bỏ qua. Dense search vẫn hoạt động ngay.
""")

BM25_FILES = [
    PROJECT_ROOT / "dataset" / "bm25" / "bm25_legal_v2.pkl",
    PROJECT_ROOT / "dataset" / "bm25" / "bm25_legal_v2.meta.pkl",
]

# Bỏ comment để xoá cache:
# for f in BM25_FILES:
#     if f.exists():
#         f.unlink()
#         print(f"  Đã xoá: {f}")

print("BM25 cache status:")
for f in BM25_FILES:
    status = f"✅ {f.stat().st_size/1024/1024:.1f} MB" if f.exists() else "❌ (chưa có)"
    print(f"  {status}  {f.name}")

📋 Hướng dẫn rebuild BM25 sau khi cập nhật ChromaDB:

  OPTION 1 — Rebuild qua hybrid_retrieval.py (khuyến nghị):
    from hybrid_retrieval import init_retriever
    init_retriever(force_rebuild_bm25=True)

  OPTION 2 — Xoá cache thủ công rồi chạy lại:
    Xoá file .pkl → lần chạy tiếp tự detect và rebuild.

  Khi nào cần rebuild ngay?
    - Upsert / xoá > 1% tổng chunks (~2,400 chunks): rebuild ngay.
    - Ít hơn: bỏ qua. Dense search vẫn hoạt động ngay.

BM25 cache status:
  ✅ 575.6 MB  bm25_legal_v2.pkl
  ✅ 22.0 MB  bm25_legal_v2.meta.pkl


In [73]:
# ============================================================
# Cell: Rebuild BM25 Index — đồng bộ từ ChromaDB → Parquet → BM25
# Chạy sau khi upsert/xóa đáng kể (> ~1% tổng chunks)
# Thời gian ước tính: 5–12 phút tùy CPU & số chunks
# ============================================================

import sys, time
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hybrid_retrieval import init_retriever, BM25_DIR, CHUNK_DIR

# ── BƯỚC 0: Kiểm tra trạng thái TRƯỚC ─────────────────────────────────────
BM25_FILES = [
    BM25_DIR / "bm25_legal_v2.pkl",
    BM25_DIR / "bm25_legal_v2.meta.pkl",
    BM25_DIR / "bm25_forms_v2.pkl",
    BM25_DIR / "bm25_forms_v2.meta.pkl",
    BM25_DIR / "bm25_examples_v2.pkl",
    BM25_DIR / "bm25_examples_v2.meta.pkl",
]
PARQUET_FILES = {
    "legal"   : CHUNK_DIR / "legal_chunks.parquet",
    "forms"   : CHUNK_DIR / "forms_chunks.parquet",
    "examples": CHUNK_DIR / "examples_chunks.parquet",
}

print("📦 Trạng thái TRƯỚC rebuild:")
for col_key, pq_path in PARQUET_FILES.items():
    if pq_path.exists():
        df_old = pd.read_parquet(pq_path)
        print(f"  parquet {col_key:<10}: {len(df_old):,} rows")
    else:
        print(f"  parquet {col_key:<10}: ❌ chưa có")

# ── BƯỚC 1: Export ChromaDB → Parquet để đồng bộ ─────────────────────────
print("\n🔄 Bước 1: Export ChromaDB → Parquet (đồng bộ chunks mới)...")

import chromadb
from chromadb.config import Settings

chroma_client = chromadb.PersistentClient(
    path=str(PROJECT_ROOT / "dataset" / "chromadb"),
    settings=Settings(anonymized_telemetry=False),
)

COLLECTION_NAMES = {
    "legal"   : "legal_chunks",
    "forms"   : "forms_chunks",
    "examples": "examples_chunks",
}

# Cột metadata cần giữ lại — khớp với LEGAL_META_COLS trong notebook + chunk_id riêng
META_COLS = [
    "doc_id", "source_doc_no", "ministry", "type_normalized",
    "doc_name", "chapter_id", "chapter_name", "article",
    "chunk_index", "total_chunks", "split_type", "word_count",
]

def export_collection_to_parquet(col_name: str, parquet_path: Path) -> pd.DataFrame:
    """Export toàn bộ ChromaDB collection ra parquet, giữ chunk_id và text."""
    col = chroma_client.get_collection(col_name)
    total = col.count()
    print(f"  [{col_name}] Exporting {total:,} chunks...")

    all_ids, all_docs, all_metas = [], [], []
    batch_size = 5000
    offset = 0

    while offset < total:
        batch = col.get(
            limit=batch_size,
            offset=offset,
            include=["documents", "metadatas"],
        )
        all_ids.extend(batch["ids"])
        all_docs.extend(batch["documents"])
        all_metas.extend(batch["metadatas"])
        offset += len(batch["ids"])
        print(f"    {offset:,}/{total:,}...", end="\r")

    print(f"    {total:,}/{total:,} ✅")

    rows = []
    for chroma_id, doc_text, meta in zip(all_ids, all_docs, all_metas):
        row = {"chunk_id": chroma_id, "text": doc_text}
        for col_key in META_COLS:
            row[col_key] = meta.get(col_key, "")
        rows.append(row)

    df = pd.DataFrame(rows)

    # Đảm bảo kiểu dữ liệu đúng
    for int_col in ["chunk_index", "total_chunks", "word_count"]:
        if int_col in df.columns:
            df[int_col] = pd.to_numeric(df[int_col], errors="coerce").fillna(0).astype(int)

    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(parquet_path, index=False)
    print(f"  [{col_name}] Saved → {parquet_path.name} ({len(df):,} rows)")
    return df

t0 = time.time()
for col_key, col_name in COLLECTION_NAMES.items():
    export_collection_to_parquet(col_name, PARQUET_FILES[col_key])

print(f"\n✅ Export hoàn tất trong {time.time()-t0:.1f}s")

# ── BƯỚC 2: Rebuild BM25 từ parquet vừa sync ──────────────────────────────
print("\n🔄 Bước 2: Rebuild BM25 index (force_rebuild_bm25=True)...")
print("   Ước tính: 3–8 phút tùy CPU.")

t1 = time.time()
init_retriever(force_rebuild_bm25=True)
elapsed = time.time() - t1
print(f"\n✅ BM25 rebuild hoàn tất trong {elapsed:.1f}s ({elapsed/60:.1f} phút)")

# ── BƯỚC 3: Kiểm tra trạng thái SAU ───────────────────────────────────────
print("\n📦 Trạng thái SAU rebuild:")
for col_key, pq_path in PARQUET_FILES.items():
    if pq_path.exists():
        df_new = pd.read_parquet(pq_path)
        print(f"  parquet {col_key:<10}: {len(df_new):,} rows")
    else:
        print(f"  parquet {col_key:<10}: ❌ (build thất bại!)")

for f in BM25_FILES:
    if f.exists():
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  ✅  {f.name:<35} {size_mb:.1f} MB")
    else:
        print(f"  ❌  {f.name:<35} (build thất bại!)")

📦 Trạng thái TRƯỚC rebuild:
  parquet legal     : 213,487 rows
  parquet forms     : 10 rows
  parquet examples  : 150 rows

🔄 Bước 1: Export ChromaDB → Parquet (đồng bộ chunks mới)...
  [legal_chunks] Exporting 213,529 chunks...
    213,529/213,529 ✅.
  [legal_chunks] Saved → legal_chunks.parquet (213,529 rows)
  [forms_chunks] Exporting 10 chunks...
    10/10 ✅.
  [forms_chunks] Saved → forms_chunks.parquet (10 rows)
  [examples_chunks] Exporting 150 chunks...
    150/150 ✅.
  [examples_chunks] Saved → examples_chunks.parquet (150 rows)

✅ Export hoàn tất trong 21.4s

🔄 Bước 2: Rebuild BM25 index (force_rebuild_bm25=True)...
   Ước tính: 3–8 phút tùy CPU.
Loading chunk files...
  legal=213,529  forms=10  examples=150
[legal] Building BM25 bigram index (213,529 docs)...
  Done in 85.9s | 575.7 MB
[forms] Building BM25 bigram index (10 docs)...
  Done in 0.0s | 0.0 MB
[examples] Building BM25 bigram index (150 docs)...
  Done in 0.1s | 0.7 MB
Building ChromaDB ID maps (FIX 1)...
  lega

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Connecting to ChromaDB...


The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.


  legal=213,529  forms=10  examples=150
Loading reranker...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Retriever v4 initialized.

✅ BM25 rebuild hoàn tất trong 108.9s (1.8 phút)

📦 Trạng thái SAU rebuild:
  parquet legal     : 213,529 rows
  parquet forms     : 10 rows
  parquet examples  : 150 rows
  ✅  bm25_legal_v2.pkl                   575.7 MB
  ✅  bm25_legal_v2.meta.pkl              5.5 MB
  ✅  bm25_forms_v2.pkl                   0.0 MB
  ✅  bm25_forms_v2.meta.pkl              0.0 MB
  ✅  bm25_examples_v2.pkl                0.7 MB
  ✅  bm25_examples_v2.meta.pkl           0.0 MB
